## 4.3 앙상블 학습 개요

### Voting Classifier

In [ ]:
# 필요한 라이브러리 임포트
import pandas as pd

# 보팅(Voting) 앙상블 분류기: 여러 분류기의 예측 결과를 결합하여 최종 예측을 수행
from sklearn.ensemble import VotingClassifier
# 개별 분류기로 사용할 로지스틱 회귀 모델
from sklearn.linear_model import LogisticRegression
# 개별 분류기로 사용할 KNN(K-Nearest Neighbors) 모델
from sklearn.neighbors import KNeighborsClassifier
# 사이킷런 내장 데이터셋: 위스콘신 유방암(Breast Cancer) 진단 데이터
from sklearn.datasets import load_breast_cancer
# 학습/테스트 데이터 분리를 위한 함수
from sklearn.model_selection import train_test_split
# 분류 정확도 평가 지표
from sklearn.metrics import accuracy_score

# 위스콘신 유방암 데이터셋 로드 (이진 분류 문제: 악성/양성)
cancer = load_breast_cancer()

# 피처 데이터를 DataFrame으로 변환 (컬럼명에 피처 이름 부여)
data_df = pd.DataFrame(cancer.data, columns=cancer.feature_names)
# 데이터 상위 3개 행 출력하여 구조 확인
data_df.head(3)


In [ ]:
# ========== 개별 분류 모델 정의 ==========
# 로지스틱 회귀 모델: solver='liblinear'는 소규모 데이터셋과 이진 분류에 적합한 최적화 알고리즘
lr_clf = LogisticRegression(solver='liblinear')
# KNN 분류기: n_neighbors=8은 예측 시 가장 가까운 8개의 이웃 데이터를 참고
knn_clf = KNeighborsClassifier(n_neighbors=8)

# ========== Voting Classifier 생성 ==========
# estimators: 결합할 개별 모델들을 (이름, 모델) 튜플의 리스트로 전달
# voting='soft': 소프트 보팅 - 각 분류기의 예측 확률(predict_proba)을 평균내어 최종 클래스 결정
#                (하드 보팅은 단순 다수결 방식이며, 일반적으로 소프트 보팅의 예측 성능이 더 우수함)
vo_clf = VotingClassifier( estimators=[('LR',lr_clf),('KNN',knn_clf)] , voting='soft' )

# 학습/테스트 데이터 분리 (8:2 비율, random_state=156으로 결과 재현성 보장)
X_train, X_test, y_train, y_test = train_test_split(cancer.data, cancer.target, 
                                                    test_size=0.2 , random_state= 156)

# ========== VotingClassifier 학습 및 평가 ==========
# 보팅 분류기 학습 (내부적으로 개별 모델들이 각자 학습됨)
vo_clf.fit(X_train , y_train)
# 학습된 보팅 분류기로 테스트 데이터에 대한 예측 수행
pred = vo_clf.predict(X_test)
# 정확도 출력 (소수점 4자리까지)
print('Voting 분류기 정확도: {0:.4f}'.format(accuracy_score(y_test , pred)))

# ========== 개별 모델의 성능 비교 ==========
# 보팅 앙상블의 효과를 확인하기 위해 개별 모델의 정확도도 함께 측정
classifiers = [lr_clf, knn_clf]
for classifier in classifiers:
    # 각 개별 모델을 학습
    classifier.fit(X_train , y_train)
    # 학습된 모델로 예측
    pred = classifier.predict(X_test)
    # 클래스 이름(모델명) 추출하여 출력에 활용
    class_name= classifier.__class__.__name__
    print('{0} 정확도: {1:.4f}'.format(class_name, accuracy_score(y_test , pred)))

## 4.4 Random Forest

In [ ]:
# Human Activity Recognition 데이터셋의 features.txt 파일에는 중복된 피처명이 존재함.
# pandas는 동일한 컬럼명이 있을 경우 DataFrame 생성 시 오류가 발생하므로,
# 중복된 피처명에 일련번호(_1, _2 등)를 붙여서 고유한 이름으로 만들어 주는 함수.
def get_new_feature_name_df(old_feature_name_df):
    # groupby('column_name').cumcount(): 동일한 컬럼명이 나타날 때마다 누적 개수를 매김
    # 처음 등장한 컬럼명은 0, 두 번째는 1, 세 번째는 2...의 dup_cnt 값을 갖게 됨
    feature_dup_df = pd.DataFrame(data=old_feature_name_df.groupby('column_name').cumcount(),
                                  columns=['dup_cnt'])
    # index를 컬럼으로 복원 (병합을 위해)
    feature_dup_df = feature_dup_df.reset_index()
    # 원본 피처명 DataFrame과 중복 카운트 DataFrame을 outer 조인으로 병합
    new_feature_name_df = pd.merge(old_feature_name_df.reset_index(), feature_dup_df, how='outer')
    # dup_cnt가 0보다 크면(즉, 중복된 경우) 컬럼명 뒤에 _숫자를 붙임. 그렇지 않으면 원래 이름 유지
    new_feature_name_df['column_name'] = new_feature_name_df[['column_name', 'dup_cnt']].apply(lambda x : x[0]+'_'+str(x[1]) 
                                                                                         if x[1] >0 else x[0] ,  axis=1)
    # 불필요한 index 컬럼 제거
    new_feature_name_df = new_feature_name_df.drop(['index'], axis=1)
    return new_feature_name_df

In [ ]:
import pandas as pd

# UCI Human Activity Recognition 데이터셋(스마트폰 센서로 수집한 사람의 행동 인식 데이터)을
# 학습/테스트용 DataFrame으로 로드하는 함수
def get_human_dataset( ):
    
    # 각 데이터 파일들은 공백으로 분리되어 있으므로 read_csv에서 공백 문자(\s+)를 sep으로 할당
    # features.txt는 헤더가 없으므로 직접 컬럼명(column_index, column_name)을 부여
    feature_name_df = pd.read_csv('./human_activity/features.txt',sep='\s+',
                        header=None,names=['column_index','column_name'])
    
    # 중복된 피처명을 수정하는 get_new_feature_name_df()를 이용해 신규 피처명 DataFrame 생성
    new_feature_name_df = get_new_feature_name_df(feature_name_df)
    
    # DataFrame에 피처명을 컬럼으로 부여하기 위해 두 번째 컬럼(피처명)만 리스트 객체로 변환
    feature_name = new_feature_name_df.iloc[:, 1].values.tolist()
    
    # 학습/테스트용 피처 데이터(X)를 DataFrame으로 로딩하면서 컬럼명은 feature_name 적용
    X_train = pd.read_csv('./human_activity/train/X_train.txt',sep='\s+', names=feature_name )
    X_test = pd.read_csv('./human_activity/test/X_test.txt',sep='\s+', names=feature_name)
    
    # 학습/테스트용 레이블 데이터(y)를 DataFrame으로 로딩하고 컬럼명은 'action'으로 부여
    # (action 값: 1~6은 걷기, 계단 오르내리기, 앉기, 서기, 눕기 등의 행동을 의미)
    y_train = pd.read_csv('./human_activity/train/y_train.txt',sep='\s+',header=None,names=['action'])
    y_test = pd.read_csv('./human_activity/test/y_test.txt',sep='\s+',header=None,names=['action'])
    
    # 로드된 학습/테스트용 DataFrame을 모두 반환
    return X_train, X_test, y_train, y_test


# 함수를 호출하여 학습/테스트 데이터셋 획득
X_train, X_test, y_train, y_test = get_human_dataset()

In [ ]:
# 랜덤 포레스트 분류기(배깅 기반의 대표적인 앙상블 알고리즘) 임포트
# - 여러 개의 결정 트리를 부트스트랩 샘플링으로 학습하고 그 결과를 다수결로 결합
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pandas as pd
import warnings
# 사이킷런 버전 호환성 등으로 발생하는 경고 메시지 출력을 무시
warnings.filterwarnings('ignore')

# 결정 트리에서 사용한 get_human_dataset()을 이용해 학습/테스트용 DataFrame 반환
X_train, X_test, y_train, y_test = get_human_dataset()

# ========== 랜덤 포레스트 학습 및 예측 성능 평가 ==========
# RandomForestClassifier 객체 생성 (random_state=0으로 결과 재현성 보장)
# 별도 옵션을 지정하지 않으면 기본값으로 n_estimators=100(트리 100개) 사용
rf_clf = RandomForestClassifier(random_state=0)
# 학습 수행
rf_clf.fit(X_train , y_train)
# 테스트 데이터에 대한 예측 수행
pred = rf_clf.predict(X_test)
# 정확도 계산
accuracy = accuracy_score(y_test , pred)
print('랜덤 포레스트 정확도: {0:.4f}'.format(accuracy))

In [ ]:
# GridSearchCV: 하이퍼 파라미터 후보 조합을 모두 시도하여 최적 조합을 찾아주는 클래스
from sklearn.model_selection import GridSearchCV

# 튜닝할 하이퍼 파라미터들의 후보 값 딕셔너리
params = {
    'n_estimators':[100],          # 트리의 개수 (앙상블에 포함할 결정 트리 수)
    'max_depth' : [6, 8, 10, 12],  # 각 트리의 최대 깊이 (과적합 방지용)
    'min_samples_leaf' : [8, 12, 18 ],   # 리프 노드가 되기 위한 최소 샘플 수
    'min_samples_split' : [8, 16, 20]    # 노드 분할을 위한 최소 샘플 수
}
# RandomForestClassifier 객체 생성. n_jobs=-1은 사용 가능한 모든 CPU 코어를 병렬로 사용
rf_clf = RandomForestClassifier(random_state=0, n_jobs=-1)
# GridSearchCV 수행: cv=2(2-fold 교차검증), n_jobs=-1로 병렬 처리
# 총 후보 조합 수: 1 * 4 * 3 * 3 = 36개, 각 조합마다 2번 학습 → 72번 학습 수행
grid_cv = GridSearchCV(rf_clf , param_grid=params , cv=2, n_jobs=-1 )
grid_cv.fit(X_train , y_train)

# 가장 우수한 성능을 보인 하이퍼 파라미터 조합과 그 때의 평균 정확도 출력
print('최적 하이퍼 파라미터:\n', grid_cv.best_params_)
print('최고 예측 정확도: {0:.4f}'.format(grid_cv.best_score_))

In [ ]:
# GridSearchCV로 찾은 최적 하이퍼 파라미터를 적용한 RandomForest 모델 학습
# - n_estimators=300: 트리 개수를 100→300으로 늘려 앙상블 안정성 강화
# - max_depth=10, min_samples_leaf=8, min_samples_split=8: GridSearch에서 도출된 최적 값
# - '\'는 파이썬 코드의 줄 바꿈을 의미함
rf_clf1 = RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_leaf=8, \
                                 min_samples_split=8, random_state=0)
# 전체 학습 데이터로 재학습
rf_clf1.fit(X_train , y_train)
# 테스트 데이터로 예측 수행
pred = rf_clf1.predict(X_test)
# 정확도 평가
print('예측 정확도: {0:.4f}'.format(accuracy_score(y_test , pred)))

In [ ]:
# 시각화를 위한 라이브러리 임포트
import matplotlib.pyplot as plt
import seaborn as sns
# 주피터 노트북에서 그래프를 인라인(셀 출력 영역)에 표시
%matplotlib inline

# 학습된 랜덤포레스트 모델의 feature_importances_ 속성을 통해 피처별 중요도 값 추출
# (각 피처가 트리 분할에 얼마나 기여했는지를 나타내는 0~1 사이의 정규화된 값)
ftr_importances_values = rf_clf1.feature_importances_
# 피처 중요도 값을 인덱스가 피처명인 Series로 변환 (시각화 및 정렬 용이)
ftr_importances = pd.Series(ftr_importances_values,index=X_train.columns  )
# 중요도가 높은 순으로 정렬하여 상위 20개 피처만 선택
ftr_top20 = ftr_importances.sort_values(ascending=False)[:20]

# Figure 크기 설정 (가로 8인치, 세로 6인치)
plt.figure(figsize=(8,6))
plt.title('Feature importances Top 20')
# seaborn의 barplot으로 가로 막대 그래프 그리기 (x: 중요도 값, y: 피처명)
sns.barplot(x=ftr_top20 , y = ftr_top20.index)
# 현재 figure 객체를 가져옴 (이후 파일로 저장하기 위해)
fig1 = plt.gcf()
# 화면에 그래프 표시
plt.show()
# 다음 plot 호출 전에 캔버스 갱신
plt.draw()
# 그래프를 TIF 이미지 파일로 저장 (300 DPI 고해상도, 여백 자동 조정)
fig1.savefig('rf_feature_importances_top20.tif', format='tif', dpi=300, bbox_inches='tight')

## 4.5 GBM(Gradient Boosting Machine)

In [ ]:
# GradientBoostingClassifier: 부스팅 계열 앙상블 알고리즘
# - 이전 트리의 오류(잔차)를 다음 트리가 학습하는 방식으로 순차적으로 모델을 강화
# - 랜덤포레스트와 달리 트리를 병렬이 아닌 순차적으로 학습하므로 학습 시간이 오래 걸림
from sklearn.ensemble import GradientBoostingClassifier
# 수행 시간 측정을 위한 time 모듈
import time
import warnings
warnings.filterwarnings('ignore')

# 학습/테스트 데이터셋 재로드
X_train, X_test, y_train, y_test = get_human_dataset()

# GBM 수행 시간 측정을 위한 시작 시간 기록 (현재 시각을 초 단위로 저장)
start_time = time.time()

# GBM 분류기 객체 생성 (random_state=0으로 결과 재현성 보장)
# 기본값: n_estimators=100, learning_rate=0.1, max_depth=3
gb_clf = GradientBoostingClassifier(random_state=0)
# 학습 수행 (순차적 트리 학습으로 인해 RandomForest보다 시간이 더 오래 소요됨)
gb_clf.fit(X_train , y_train)
# 학습된 모델로 테스트 데이터 예측
gb_pred = gb_clf.predict(X_test)
# 정확도 계산
gb_accuracy = accuracy_score(y_test, gb_pred)

print('GBM 정확도: {0:.4f}'.format(gb_accuracy))
# 종료 시간에서 시작 시간을 빼서 총 수행 시간(초) 출력
print("GBM 수행 시간: {0:.1f} 초 ".format(time.time() - start_time))


In [ ]:
### 아래는 책에서 설명드리지는 않지만 GridSearchCV로 GBM의 하이퍼 파라미터 튜닝을 수행하는 예제 입니다. 
### 사이킷런이 1.X로 업그레이드 되면서 GBM의 학습 속도가 현저하게 저하되는 문제가 오히려 발생합니다. 
### 아래는 수행 시간이 오래 걸리므로 참고용으로만 사용하시면 좋을 것 같습니다. 

from sklearn.model_selection import GridSearchCV

# GBM에서 주요한 하이퍼 파라미터 후보 정의
params = {
    'n_estimators':[100, 500],        # 약한 학습기(weak learner)의 개수. 클수록 성능 향상 가능성이 있지만 과적합 위험 증가
    'learning_rate' : [ 0.05, 0.1]    # 학습률. 각 트리가 이전 트리의 오류를 보정하는 비율 (낮을수록 안정적이지만 더 많은 트리 필요)
}
# GBM에 대해 GridSearchCV 수행
# cv=2 (2-fold 교차검증), verbose=1로 진행 상황 출력
# 총 후보 조합: 2 * 2 = 4개, 각 조합마다 2-fold → 총 8회 학습 (매우 오래 걸림)
grid_cv = GridSearchCV(gb_clf , param_grid=params , cv=2 ,verbose=1)
grid_cv.fit(X_train , y_train)
# 최적 하이퍼 파라미터와 그 때의 평균 교차검증 정확도 출력
print('최적 하이퍼 파라미터:\n', grid_cv.best_params_)
print('최고 예측 정확도: {0:.4f}'.format(grid_cv.best_score_))

In [ ]:
# GridSearchCV의 best_estimator_ 속성은 최적 하이퍼 파라미터로 전체 학습 데이터에 대해
# 재학습된 모델 객체를 의미함. 이를 사용해 테스트 데이터에 대한 최종 예측 수행
gb_pred = grid_cv.best_estimator_.predict(X_test)
# 테스트 데이터에 대한 최종 정확도 계산
gb_accuracy = accuracy_score(y_test, gb_pred)
print('GBM 정확도: {0:.4f}'.format(gb_accuracy))